---
title: "Chapter 5: More Preprocessing: Categorical Features and Text"
---

::: {.callout-lo}

By the end of this chapter, you will be able to:

- Explain what `handle_unknown="ignore"` does and what information it loses.
- Recognize the trade-offs of binary encoding and high-cardinality features.
- Diagnose common preprocessing mistakes in an ML workflow.
- Use `CountVectorizer` to create bag-of-words features inside a pipeline.
- Explain how vocabulary choices and sparse storage affect text representations.
:::

Chapter 4 built a complete pipeline for numeric, nominal, ordinal, and binary features. Here we examine choices that become important when that pipeline meets new data: unseen categories, features with many possible values, and mistakes that can survive even when the code runs. We then extend the same workflow to text using bag-of-words features.

**Imports**

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC

## Dealing with unknown categories

An encoder learns categories from its training data. A rare category may occur only in a validation fold, and a new category can also appear after deployment. The default `OneHotEncoder` raises an error when asked to transform a category it has not seen. We can reproduce that situation without fitting a predictive model.

In [2]:
training_majors = pd.DataFrame({"major": ["Computer Science", "History", "History"]})
new_majors = pd.DataFrame({"major": ["Biology", "Physics", "History"]})
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(training_majors)
try:
    encoder.transform(new_majors)
except ValueError as error:
    print(error)

Found unknown categories ['Physics', 'Biology'] in column 0 during transform


Setting `handle_unknown="ignore"` keeps the learned feature space fixed. An unseen major is represented by zeros in every learned `major` indicator column. This prevents an error, but it also makes all unseen majors indistinguishable from one another.

In [3]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(training_majors)
pd.DataFrame(
    encoder.transform(new_majors),
    columns=encoder.get_feature_names_out(),
    index=new_majors["major"],
)

,major_Computer Science,major_History
major,,
Biology,0.0,0.0
Physics,0.0,0.0
History,0.0,1.0


The transformation now completes, but `ignore` is a modelling choice rather than a universal fix. If unseen categories will be common or need to be distinguished, we might instead group rare categories, redesign the feature, or use an encoder that explicitly represents infrequent values.

### Categories known independently of the data

Sometimes the complete set of categories is defined independently of the observed dataset—for example, Canadian provinces and territories. Supplying such a published list does not use information from validation or test examples, so it does not violate the Golden Rule. This differs from inspecting the full dataset to discover which categories happen to occur.

### Categorical features with two possible categories

One-hot encoding a two-category feature normally creates two complementary columns. A single indicator contains the same information: knowing that one category is absent tells us that the other is present. `OneHotEncoder(drop="if_binary")` therefore emits one column for binary features while retaining all columns for features with more than two categories.

In [4]:
binary_values = pd.DataFrame({"enjoy_course": ["Yes", "No", "Yes"]})
binary_encoder = OneHotEncoder(drop="if_binary", sparse_output=False)
pd.DataFrame(
    binary_encoder.fit_transform(binary_values),
    columns=binary_encoder.get_feature_names_out(),
)

,enjoy_course_Yes
0,1.0
1,0.0
2,1.0


Dropping an indicator is optional, not a requirement for every model. It changes distances and can affect model behaviour. Also, combining a dropped category with `handle_unknown="ignore"` makes an unseen value and the dropped category share the all-zero representation. Decide whether that ambiguity is acceptable for your problem.

## High-cardinality features

One-hot encoding creates one feature per category. When a feature has many categories, some indicators may occur too rarely for the model to learn a reliable pattern. Depending on the problem, we might group categories using domain knowledge—for example, countries into regions—or combine infrequent values into an `other` category. Such grouping changes the information available to the model and should be justified rather than applied mechanically.

A feature with many distinct values is called **high cardinality**. A postal code may contain useful geographic information, whereas a unique transaction ID may offer little that generalizes to a new transaction. Replacing one-hot encoding with arbitrary integer codes saves columns but invents distances; it does not solve that modelling problem. Sparse storage can reduce memory use, but it cannot make rare categories more informative. Any frequency-based grouping must be learned from the training portion of each fold.


### Deciding whether a feature belongs in the model

A feature being available does not mean it belongs in a model. Sensitive attributes such as race or gender, and variables that act as proxies for them, require careful consideration of the application, applicable rules, measurement quality, fairness goals, and consequences of errors. Simply dropping a sensitive column does not guarantee that the model will be fair, because other features may retain closely related information.

### Preprocessing the target

In classification, `scikit-learn` estimators can usually accept string class labels directly, so one-hot encoding the target is unnecessary. Regression targets are sometimes transformed—for example, to reduce strong skew—but that transformation must be inverted when predictions are interpreted. Later chapters revisit target transformations where they are useful.

## Common preprocessing mistakes

A pipeline coordinates the steps we specify, but it cannot decide whether those steps make sense. Check both the data boundaries and the feature meanings.

| Mistake | Why it matters | Better workflow |
|---|---|---|
| Fit an imputer, scaler, encoder, or vocabulary before splitting | Test examples influence the representation | Split first and fit on training data only |
| Transform all of `X_train` before cross-validation | Each fold's validation data influence fitted preprocessing | Pass raw features and the complete pipeline to cross-validation |
| Fit separate encoders on training and test data | The same column position may represent different categories | Transform new data with the fitted training encoder |
| Give nominal categories arbitrary integer codes | A distance-based model interprets an invented ordering and spacing | Use one-hot encoding unless an order is meaningful |
| Scale numeric identifiers as if they were measurements | Numerical proximity between IDs usually has no useful meaning | Investigate the feature's role and usually omit a pure identifier |
| Include the target or information recorded after the outcome | Validation may reward information unavailable at prediction time | Define when a prediction is made and retain only available features |

A preprocessing choice should also match the estimator: scaling is usually important for $k$-NN and RBF SVMs, whereas decision trees generally do not need it. Do not infer that a transformation is useful simply because it runs.

## Encoding text data

Text cannot enter the estimators we have used so far in its raw string form. We need a transformer that learns a vocabulary from the training documents and represents each document numerically. As with every learned preprocessing step, vocabulary construction must happen inside the training workflow.

### From messages to bag-of-words features

Consider a small spam-classification dataset in which `sms` contains the message and `target` records whether it is spam. The message is neither a numeric measurement nor one value from a fixed categorical vocabulary.

In [5]:
toy_spam = [
    [
        "URGENT!! URGENT!! As a valued network customer you have been selected to receive a £900 prize reward!",
        "spam",
    ],
    ["Lol you are always so convincing.", "non spam"],
    ["Nah I don't think he goes to usf, he lives around here though", "non spam"],
    [
        "URGENT! You have won a 1 week FREE membership in our £100000 prize Jackpot!",
        "spam",
    ],
    [
        "Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030",
        "spam",
    ],
    ["Congrats! I can't wait to see you!!", "non spam"],
]
toy_df = pd.DataFrame(toy_spam, columns=["sms", "target"])
toy_df

,sms,target
0,URGENT!! URGENT!! As a valued network customer...,spam
1,Lol you are always so convincing.,non spam
2,"Nah I don't think he goes to usf, he lives aro...",non spam
3,URGENT! You have won a 1 week FREE membership ...,spam
4,Had your mobile 11 months or more? U R entitle...,spam
5,Congrats! I can't wait to see you!!,non spam


A natural first thought is to treat each message as a categorical value and one-hot encode it. The following deliberately inappropriate example shows why that representation does not generalize.

In [6]:
### DO NOT DO THIS.
enc = OneHotEncoder(sparse_output=False)
transformed = enc.fit_transform(toy_df[["sms"]])
pd.DataFrame(transformed, columns=enc.categories_)

,Congrats! I can't wait to see you!!,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,Lol you are always so convincing.,"Nah I don't think he goes to usf, he lives around here though",URGENT! You have won a 1 week FREE membership in our £100000 prize Jackpot!,URGENT!! URGENT!! As a valued network customer you have been selected to receive a £900 prize reward!
0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0


One-hot encoding complete messages treats each distinct message as an unrelated category. Because most messages occur only once, the resulting columns provide almost no reusable information for a new message. We need features based on smaller units that recur across documents.

Representing language computationally is a central problem in natural language processing (NLP). Common representations include bag-of-words counts, TF–IDF features, and learned embeddings. We begin with bag-of-words because its features are concrete and easy to inspect.

A **bag-of-words** representation instead learns a vocabulary of tokens from the training documents, then represents each document by the count—or sometimes simply the presence—of each vocabulary token. The representation discards word order and most syntax: it records which words are in the bag, not how they are arranged.

![](img/bag-of-words.png)

[Source](https://web.stanford.edu/~jurafsky/slp3/4.pdf)       

`CountVectorizer` implements this representation in `scikit-learn`. It converts a collection of documents into a document–term matrix: each row is a document, each column is a vocabulary token learned during `fit`, and each entry counts how often that token occurs in the document.

::: {.callout-note}
In the natural language processing (NLP) community, a collection of text documents is called a **corpus** (plural: corpora).
:::

In [7]:
vec = CountVectorizer()
X_counts = vec.fit_transform(toy_df["sms"])
bow_df = pd.DataFrame(
    X_counts.toarray(), columns=vec.get_feature_names_out(), index=toy_df["sms"]
)
bow_df

,08002986030,100000,11,900,always,are,around,as,been,call,...,update,urgent,usf,valued,wait,week,with,won,you,your
sms,,,,,,,,,,,,,,,,,,,,,
URGENT!! URGENT!! As a valued network customer you have been selected to receive a £900 prize reward!,0,0,0,1,0,0,0,1,1,0,...,0,2,0,1,0,0,0,0,1,0
Lol you are always so convincing.,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
"Nah I don't think he goes to usf, he lives around here though",0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
URGENT! You have won a 1 week FREE membership in our £100000 prize Jackpot!,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,1,1,0
Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,1,0,1,0,0,0,0,0,0,1,...,2,0,0,0,0,0,1,0,0,1
Congrats! I can't wait to see you!!,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0


::: {.callout-important}
`CountVectorizer` expects a one-dimensional sequence of documents, so we pass a pandas Series rather than a one-column DataFrame. If a dataset has multiple text columns that should have separate vocabularies, each column needs its own vectorizer branch in a `ColumnTransformer`.
:::

In [8]:
X_counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 71 stored elements and shape (6, 61)>

Most vocabulary words do not occur in any one document, so most entries in a document–term matrix are zero. `CountVectorizer` therefore returns a **sparse matrix**, which stores only nonzero values and their locations. Sparse storage has some bookkeeping overhead, but it can save substantial memory and computation when the fraction of nonzero entries is small.

In [9]:
print("The total number of elements: ", np.prod(X_counts.shape))
print("The number of non-zero elements: ", X_counts.nnz)
print(
    "Proportion of non-zero elements: %0.4f" % (X_counts.nnz / np.prod(X_counts.shape))
)
print(
    "The value at cell 3,%d is: %d"
    % (vec.vocabulary_["jackpot"], X_counts[3, vec.vocabulary_["jackpot"]])
)

The total number of elements:  366
The number of non-zero elements:  71
Proportion of non-zero elements: 0.1940
The value at cell 3,27 is: 1


::: {.callout-reflection}
`StandardScaler` normally subtracts the mean of each feature. What would happen to a sparse matrix if centring turned most of its zeros into nonzero values?
:::

Centring would destroy the matrix's sparsity and could require far more memory. When scaling sparse features, use a method that preserves zeros; for `StandardScaler`, this means setting `with_mean=False`. Word-count features are not routinely standardized, however, and transformations designed specifically for term counts are another option beyond this introduction.

::: {.callout-note title="One-hot encoding can also produce sparse features"}

By default, `OneHotEncoder` returns a sparse matrix. This representation can save considerable memory when the encoded feature matrix contains many categories and therefore many zeros.

For a small number of categories, a dense array is often more convenient and the memory difference is usually unimportant. Use `sparse_output=False` to request a dense result.

:::

### Controlling the representation

`CountVectorizer` exposes choices about both the vocabulary and the values stored in the matrix. `binary=True` records presence or absence instead of counts. `min_df` removes tokens that appear in too few documents, while `max_df` removes tokens that appear in too many. `max_features` limits the vocabulary size, and `ngram_range` allows sequences of adjacent words as well as individual words.

These are modelling choices rather than generic cleanup settings: each one determines which distinctions the estimator can use.

Each cell displays the number of times a particular word occurs in that message. For example, the word 'urgent' appears twice in the first message, so the count is 2.

In [10]:
bow_df.iloc[0]['urgent']

np.int64(2)

When we use `binary=True`, the representation uses presence/absence of words instead of word counts.   

In [11]:
vec_binary = CountVectorizer(binary=True)
X_counts = vec_binary.fit_transform(toy_df["sms"])
bow_df = pd.DataFrame(
    X_counts.toarray(), columns=vec_binary.get_feature_names_out(), index=toy_df["sms"]
)
bow_df

,08002986030,100000,11,900,always,are,around,as,been,call,...,update,urgent,usf,valued,wait,week,with,won,you,your
sms,,,,,,,,,,,,,,,,,,,,,
URGENT!! URGENT!! As a valued network customer you have been selected to receive a £900 prize reward!,0,0,0,1,0,0,0,1,1,0,...,0,1,0,1,0,0,0,0,1,0
Lol you are always so convincing.,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
"Nah I don't think he goes to usf, he lives around here though",0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
URGENT! You have won a 1 week FREE membership in our £100000 prize Jackpot!,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,1,0,1,1,0
Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,1,0,1,0,0,0,0,0,0,1,...,1,0,0,0,0,0,1,0,0,1
Congrats! I can't wait to see you!!,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0


Now each cell displays whether a particular word occurs in that message. For example, the word 'urgent' appears twice in the first message, so the representation is 1.

In [12]:
bow_df.iloc[0]['urgent']

np.int64(1)

We can limit the number of columns in the feature matrix using `max_features`. On a real corpus, this can control memory use and remove very low-ranked vocabulary terms.

In [13]:
vec8 = CountVectorizer(max_features=8)
X_counts = vec8.fit_transform(toy_df["sms"])
bow_df = pd.DataFrame(
    X_counts.toarray(), columns=vec8.get_feature_names_out(), index=toy_df["sms"]
)
bow_df

,free,have,mobile,the,to,update,urgent,you
sms,,,,,,,,
URGENT!! URGENT!! As a valued network customer you have been selected to receive a £900 prize reward!,0,1,0,0,1,0,2,1
Lol you are always so convincing.,0,0,0,0,0,0,0,1
"Nah I don't think he goes to usf, he lives around here though",0,0,0,0,1,0,0,0
URGENT! You have won a 1 week FREE membership in our £100000 prize Jackpot!,1,1,0,0,0,0,1,1
Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,2,0,2,2,2,2,0,0
Congrats! I can't wait to see you!!,0,0,0,0,1,0,0,1


Before counting, `CountVectorizer` converts text to lowercase by default and uses a token pattern that keeps tokens containing at least two word characters. This excludes most punctuation and single-character tokens. These defaults also determine which distinctions the representation preserves and should be reconsidered when they do not suit the application.

### Text preprocessing inside a pipeline

The vectorizer should be part of the predictive pipeline rather than fitted on the complete dataset in advance. Its vocabulary is learned from data, so it must obey the same fold boundaries as every other fitted transformation.

In [14]:
pipe = make_pipeline(CountVectorizer(), SVC())

In [15]:
pipe.fit(toy_df["sms"], toy_df["target"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('countvectorizer', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['non spam','spam']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


In [16]:
pipe.predict(toy_df["sms"])

array(['spam', 'non spam', 'non spam', 'spam', 'spam', 'non spam'],
      dtype=object)

When the pipeline is fit, `CountVectorizer` learns a vocabulary from the training documents and the SVM learns from the resulting columns. During cross-validation, each fold may therefore learn a slightly different vocabulary. A word found only in a validation, test, or deployment document is ignored rather than creating a new column or raising an error. This keeps the feature space consistent with the data used to fit the estimator.

Bag-of-words still discards word order, syntax, and much of compositional meaning. For example, it has difficulty distinguishing phrases containing the same words in different arrangements. Despite these limitations, word occurrence patterns can provide strong signals for tasks such as spam filtering. Bag-of-words is therefore a useful baseline, not a complete representation of language; representations that preserve more context and meaning are beyond the scope of this chapter.

## Exercises

### Exercise 5.1: A new category

An encoder learns the categories `red` and `blue` with `handle_unknown="ignore"`. What representation will it produce for `green` and for `yellow`? Can a downstream model distinguish those two values using this feature? How does your answer change if the encoder also drops one known category?

### Exercise 5.2: A misleading workflow

A customer table includes a unique customer ID, a province, and a comment. A student ordinally encodes all three columns and scales the result before splitting. Identify the problems and propose a representation and evaluation workflow for each feature.

### Exercise 5.3: What does the bag retain?

With default `CountVectorizer` settings, compare the representations of "dogs chase cats" and "cats chase dogs". What distinction is lost? What happens to a word found only in a validation document? Explain why each cross-validation fold may learn a different vocabulary.

## Summary

Unknown-category handling keeps the learned columns fixed, but it does not give a model knowledge about new categories. High-cardinality features need careful thought about useful information, rare values, and memory. Pipelines protect fitted preprocessing within an evaluation workflow; feature meaning and prediction-time availability still require our judgment.

Bag-of-words extends preprocessing to text by counting vocabulary tokens in each document. Keep vocabulary learning inside the predictive pipeline, and remember that this representation loses word order and much of the meaning conveyed by context.